# 진짜 최종 구현 코드


1. 문제의 본질
원래 amount는 
- offer completed 시점에 붙어있던 값임
- 근데 이 값이 항상 위 조건을 따르진 않았음

2. 1차 해결 로직 : completion_amount_current
offer received를 시작 시간으로 보고 offer completed가 되는 시간까지 사이의 amount를 더한 값

3. 2차 해결 로직 : completion_amount_prev_received
1차 해결 로직으로 구할 수 없는 경우(76건)
- 같은 프로모션(offer)가 한 번 더 온 경우
- received(1) -> amount -> received(2) -> viewed -> completed = amount의 경우
    1차 로직으로는 received(2)부터 completed 까지의 amount를 더함.
    이렇게 되면 만약 received(1)에 대한 completed일 경우 received(1)와 received(2) 사이 발생한 amount를 제외시키게됨 -> difficulty 미충족
- 이를 해결하기 위해 received(2) 이전의 received(1)을 기준으로 계산하는 로직 만듬

4. 최종 df는?
2가지 로직에 충족한 amount를 맞게 적용
- 기본은 completion_amount_current으로
- 부족한 경우 completion_amount_prev_received를 적용


In [1]:
import numpy as np
import pandas as pd

promotion_df = pd.read_csv('./data/promotion_df.csv')
merged_df = pd.read_csv('./data/all_merged.csv')


In [2]:
# amount_raw 기준으로 difficulty를 못 넘는 정상 completed 행 수
raw_under_diff = promotion_df[
    (promotion_df['is_completed'].eq(1)) &
    (promotion_df['is_normal_flow'].eq(1)) &
    (promotion_df['amount'] < promotion_df['difficulty'])
].copy()

print('amount_raw 기준 difficulty 미달 건수:', len(raw_under_diff))


amount_raw 기준 difficulty 미달 건수: 3301


In [3]:
# 각 offer_id에 대해 difficulty를 초과하는 거래가 있는지 확인

transactions = merged_df[merged_df['event'] == 'transaction'].copy()
transactions = transactions[['person', 'time', 'amount']]
transactions = transactions.sort_values(['person', 'time']).reset_index(drop=True)

views = merged_df[merged_df['event'] == 'offer viewed'].copy()
views = views[['person', 'offer_id', 'time']]
views = views.sort_values(['person', 'offer_id', 'time']).reset_index(drop=True)

tx_by_person = {}
for _, row in transactions.iterrows():
    person = row['person']
    if person not in tx_by_person:
        tx_by_person[person] = []
    tx_by_person[person].append((row['time'], row['amount']))

view_by_person_offer = {}
for _, row in views.iterrows():
    key = (row['person'], row['offer_id'])
    if key not in view_by_person_offer:
        view_by_person_offer[key] = []
    view_by_person_offer[key].append(row['time'])



In [4]:
promotion_df = promotion_df.sort_values(
    ['person', 'offer_id', 'offer received', 'offer completed']
).reset_index(drop=True)


In [5]:
completion_amount_current_list = []

for _, row in promotion_df.iterrows():
    person = row['person']
    start_time = row['offer received']
    end_time = row['offer completed']

    total = 0.0
    if person in tx_by_person:
        for tx_time, amount in tx_by_person[person]:
            if tx_time >= start_time and tx_time <= end_time:
                total += amount

    completion_amount_current_list.append(round(total, 2))

promotion_df['completion_amount_current'] = completion_amount_current_list


In [ ]:
promo_influenced_amount_list = []

for _, row in promotion_df.iterrows():
    person = row['person']
    offer_id = row['offer_id']
    viewed_time = row['offer viewed']
    end_time = row['offer completed']

    total = 0.0
    if pd.notna(viewed_time) and person in tx_by_person:
        for tx_time, amount in tx_by_person[person]:
            if tx_time >= viewed_time and tx_time <= end_time:
                total += amount

    if pd.isna(viewed_time):
        promo_influenced_amount_list.append(np.nan)
    else:
        promo_influenced_amount_list.append(round(total, 2))

promotion_df['promo_influenced_amount'] = promo_influenced_amount_list


In [20]:
display(
    promotion_df[
        [
            'person', 'offer_id', 'offer_cycle',
            'offer received', 'offer viewed', 'offer completed',
            'difficulty', 'completion_amount_current',
            'promo_influenced_amount'
        ]
    ].head(20)
)


,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,difficulty,completion_amount_current,promo_influenced_amount
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,Bogo_1,408.0,456.0,414.0,5,8.57,0.00
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,Discount_1,504.0,540.0,528.0,10,14.11,0.00
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,Discount_1,576.0,NaN,576.0,10,10.27,NaN
3,0009655768c64bdeb2e877511632db8f,informational_0_0_3,Informational_1,168.0,192.0,NaN,0,0.00,0.00
4,0009655768c64bdeb2e877511632db8f,informational_0_0_4,Informational_1,336.0,372.0,NaN,0,0.00,0.00
5,00116118485d4dfda04fdbaba9a87b5c,bogo_5_5_5,Bogo_1,168.0,216.0,NaN,5,0.00,0.00
6,00116118485d4dfda04fdbaba9a87b5c,bogo_5_5_5,Bogo_2,576.0,630.0,NaN,5,0.00,0.00
7,0011e0d4e6b944f998e987f904e8c1e5,bogo_5_5_7,Bogo_1,504.0,516.0,576.0,5,22.05,22.05
8,0011e0d4e6b944f998e987f904e8c1e5,discount_20_5_10,Discount_1,408.0,432.0,576.0,20,22.05,22.05
9,0011e0d4e6b944f998e987f904e8c1e5,discount_7_3_7,Discount_1,168.0,186.0,252.0,7,11.93,11.93


In [22]:
anomaly = promotion_df[
    (promotion_df['is_completed'].eq(1)) &
    (promotion_df['is_normal_flow'].eq(1)) &
    (promotion_df['completion_amount_current'] < promotion_df['difficulty'])
].copy()

print('anomaly 행 수:', len(anomaly))


anomaly 행 수: 76


In [ ]:
# # 저장
# promotion_final.to_csv('./data/promotion_final.csv', index=False)


# `amount` 재계산 및 보정 정리

이 노트북에서는 `promotion_df.csv`를 기준으로 offer row의 `amount`를 다시 계산했다.

## 작업 내용
- `offer received`부터 `offer completed`까지의 transaction 금액을 다시 합쳐 `completion_amount_current`를 계산했다.
- 현재 received 기준 금액이 `difficulty`를 못 넘는 정상 흐름 행만 따로 뽑았다.
- `offer viewed` 이후의 프로모션 영향액도 `promo_influenced_amount`로 함께 계산했다.
- 최종적으로 `adjusted_completion_amount`를 만들고, `amount_raw`는 원본값으로 보존했다.
- 저장용 최종 파일은 `promotion_final.csv`로 만들었다.

## 핵심 정리
- `completion_amount_current` = 현재 received 기준 총 결제액
- `promo_influenced_amount` = viewed 이후 결제액
- `adjusted_completion_amount` = 최종 보정된 amount

## 해석
- `current`가 충분하면 그대로 사용했다.
- 정상 흐름의 offer row를 더 정확하게 해석할 수 있도록 amount를 다시 정리한 작업이다.
